# Homework 2 (Module 2 — 2026 Cohort)
**Course:** [Stock Markets Analytics Zoomcamp](https://github.com/DataTalksClub/stock-markets-analytics-zoomcamp)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ivazin/stock-markets-analytics-zoomcamp/blob/main/homeworks/hw02/main.ipynb)

This notebook contains complete solutions and code for Homework 2 (IPOs, Financial Metrics, Holding Strategies, and RSI Technical Strategy).

## Environment & Dependencies Setup
Install required packages if running in Google Colab or a new virtual environment.

In [1]:
# Install necessary dependencies (especially for Google Colab)
!pip install -q yfinance curl_cffi gdown pyarrow lxml

import io
import re
import os
import numpy as np
import pandas as pd
import yfinance as yf
from curl_cffi import requests
import gdown

pd.set_option("display.max_columns", 30)
pd.set_option("display.precision", 4)
print("Libraries imported successfully!")

zsh:1: command not found: pip
Libraries imported successfully!


---
## Question 1: [IPO] Withdrawn IPOs by Company Type

**What is the total withdrawn IPO value (in $ millions) for the company class with the highest total withdrawal value?**

### Steps:
1. Scrape `https://www.iposcoop.com/ipos-recently-filed/` and filter rows where `Expected To Trade` is `Withdrawn`.
2. Classify companies by name according to strict priority order:
   - `Technologies`
   - `Acquisition Corp`, `Acquisition Corporation`, or `Corp` -> `Acquisition Corp`
   - `Inc` or `Incorporated` -> `Inc.`
   - `Group`
   - `Ltd` or `Limited` -> `Limited`
   - `Holdings` or `Holding` -> `Holdings`
   - Others -> `Other`
3. Calculate `Avg_price` from `Price Low` and `Price High`.
4. Compute `Shares_offered_value`: use `Shares (millions) * Avg_price` if available, otherwise fall back to `Est $ Vol (millions)`.
5. Aggregate by `Company Type`.

In [2]:
# Scrape recently filed IPO table using curl_cffi to bypass Sucuri Cloudproxy
url_filed = "https://www.iposcoop.com/ipos-recently-filed/"
resp = requests.get(url_filed, impersonate="chrome")
df_recently_filed = pd.read_html(io.StringIO(resp.text))[0]

# Filter for Withdrawn entries
df_w = df_recently_filed[df_recently_filed["Expected To Trade"] == "Withdrawn"].copy()
print(f"Total Withdrawn IPOs found: {len(df_w)}")

# 1. Company classification
def classify_company(name: str) -> str:
    if re.search(r"\bTechnologies\b", name):
        return "Technologies"
    if re.search(r"\b(Acquisition Corp|Acquisition Corporation|Corp)\b", name):
        return "Acquisition Corp"
    if re.search(r"\b(Inc|Incorporated)\b", name):
        return "Inc."
    if re.search(r"\bGroup\b", name):
        return "Group"
    if re.search(r"\b(Ltd|Limited)\b", name):
        return "Limited"
    if re.search(r"\b(Holdings|Holding)\b", name):
        return "Holdings"
    return "Other"

df_w["Company Type"] = df_w["Company"].apply(classify_company)

# 2. Currency/number parser
def parse_currency(val):
    if pd.isna(val):
        return None
    val_clean = str(val).replace("$", "").replace(",", "").strip()
    try:
        return float(val_clean)
    except ValueError:
        return None

df_w["price_low_num"] = df_w["Price Low"].apply(parse_currency)
df_w["price_high_num"] = df_w["Price High"].apply(parse_currency)
df_w["Avg_price"] = (df_w["price_low_num"] + df_w["price_high_num"]) / 2.0
df_w["shares_num"] = df_w["Shares (millions)"].apply(parse_currency)
df_w["est_vol_num"] = df_w["Est $ Vol (millions)"].apply(parse_currency)

# 3. Calculate Shares_offered_value
calc_val = df_w["shares_num"] * df_w["Avg_price"]
df_w["Shares_offered_value"] = calc_val.combine_first(df_w["est_vol_num"])

# 4. Aggregate sum by Company Type
agg_all = df_w.groupby("Company Type")["Shares_offered_value"].sum().sort_values(ascending=False)
print("\n--- Total Withdrawn Value by Company Type (All entries) ---")
display(agg_all.to_frame())

# Also inspect for entries filed before Sep 11, 2026
df_w_sep11 = df_w[df_w["File Date"] < "2026-09-11"].copy()
agg_sep11 = df_w_sep11.groupby("Company Type")["Shares_offered_value"].sum().sort_values(ascending=False)
print(f"\n--- Total Withdrawn Value by Company Type (Before Sep 11, 2026: {len(df_w_sep11)} entries) ---")
display(agg_sep11.to_frame())

print(f"\nAnswer Q1: {agg_all.index[0]} with ${agg_all.iloc[0]:.2f}M")

Total Withdrawn IPOs found: 34

--- Total Withdrawn Value by Company Type (All entries) ---


,Shares_offered_value
Company Type,
Acquisition Corp,499.9850
Inc.,351.0000
Holdings,311.6575
Other,290.4450
Limited,219.2500
Technologies,184.9000
Group,56.5750



--- Total Withdrawn Value by Company Type (Before Sep 11, 2026: 31 entries) ---


,Shares_offered_value
Company Type,
Acquisition Corp,499.9850
Inc.,351.0000
Holdings,311.6575
Other,290.4450
Limited,197.2500
Technologies,184.9000
Group,32.5000



Answer Q1: Acquisition Corp with $499.99M


---
## Question 2: [IPO] Median Sharpe Ratio for 2025 IPOs (First 8 Months)

**What is the median Sharpe ratio (as of 11 September 2026) for companies that went public before 1 September 2025?**

### Steps:
1. Download 2025 IPOs list from `https://www.iposcoop.com/2025-pricings/`.
2. Filter IPOs with `Offer Date < 2025-09-01` and non-zero returns (`Return != "0.00%"`).
3. Download daily stock data with `yfinance` for active tickers.
4. Compute:
   - `growth_252d = Close / Close.shift(252)`
   - `volatility = Close.rolling(30).std() * sqrt(252)`
   - `Sharpe = (growth_252d - 0.05) / volatility` (assuming 5% risk-free rate).
5. Filter for trading date `2026-09-11` and compute median Sharpe ratio.

In [3]:
# Load 2025 pricings table
url_pricings = "https://www.iposcoop.com/2025-pricings/"
resp_p = requests.get(url_pricings, impersonate="chrome")
df_pricings = pd.read_html(io.StringIO(resp_p.text))[0]

df_pricings["Offer Date Parsed"] = pd.to_datetime(df_pricings["Offer Date"], errors="coerce")
filtered_ipos = df_pricings[(df_pricings["Offer Date Parsed"] < "2025-09-01") & (df_pricings["Return"] != "0.00%")].copy()
tickers = filtered_ipos["Symbol"].dropna().unique().tolist()
tickers_clean = [t.strip().replace(".", "-") for t in tickers]
print(f"Target tickers count: {len(tickers_clean)}")

# Download or load cached stocks_df
stocks_parquet = "homeworks/hw02/stocks_df.parquet" if os.path.exists("homeworks/hw02/stocks_df.parquet") else "stocks_df.parquet"
if os.path.exists(stocks_parquet):
    print(f"Loading cached {stocks_parquet}...")
    stocks_df = pd.read_parquet(stocks_parquet)
else:
    print("Downloading daily stock data from Yahoo Finance...")
    records = []
    for t in tickers_clean:
        try:
            t_df = yf.download(t, start="2025-01-01", end="2026-09-15", progress=False, auto_adjust=False)
            if t_df is not None and not t_df.empty:
                if isinstance(t_df.columns, pd.MultiIndex):
                    t_df.columns = t_df.columns.get_level_values(0)
                t_df["Ticker"] = t
                records.append(t_df)
        except Exception:
            pass
    stocks_df = pd.concat(records).reset_index()
    stocks_df["Date"] = pd.to_datetime(stocks_df["Date"])
    stocks_df.to_parquet("stocks_df.parquet")

stocks_df = stocks_df.sort_values(["Ticker", "Date"]).reset_index(drop=True)
print(f"Total unique tickers in dataset: {stocks_df['Ticker'].nunique()}")

# Feature Engineering
stocks_df["growth_252d"] = stocks_df.groupby("Ticker")["Close"].shift(0) / stocks_df.groupby("Ticker")["Close"].shift(252)
stocks_df["volatility"] = stocks_df.groupby("Ticker")["Close"].transform(lambda s: s.rolling(30).std() * np.sqrt(252))
stocks_df["Sharpe"] = (stocks_df["growth_252d"] - 0.05) / stocks_df["volatility"]

# Filter for 2026-09-11
df_sep11 = stocks_df[stocks_df["Date"].dt.strftime("%Y-%m-%d") == "2026-09-11"].copy()
print("\n--- Descriptive Statistics as of 2026-09-11 ---")
display(df_sep11[["growth_252d", "volatility", "Sharpe"]].describe())

median_sharpe = df_sep11["Sharpe"].median()
print(f"\nAnswer Q2: Median Sharpe Ratio as of 2026-09-11 = {median_sharpe:.4f} (~0.05)")

Target tickers count: 146
Loading cached stocks_df.parquet...
Total unique tickers in dataset: 132

--- Descriptive Statistics as of 2026-09-11 ---


/Users/user/Documents/Python/pythoninvest-stock-markets-analytics-zoomcamp/.venv/lib/python3.14/site-packages/pandas/core/nanops.py:1028: RuntimeWarning: invalid value encountered in subtract
  sqr = _ensure_numeric((avg - values) ** 2)


Price,growth_252d,volatility,Sharpe
count,130.0000,131.0000,130.0000
mean,1.0521,27.5720,inf
std,3.0492,66.8680,NaN
min,0.0010,0.0000,-0.0401
25%,0.1410,2.3013,0.0120
50%,0.5945,7.8857,0.0490
75%,1.0388,23.9685,0.1336
max,33.6383,627.6981,inf



Answer Q2: Median Sharpe Ratio as of 2026-09-11 = 0.0490 (~0.05)


---
## Question 3: [IPO] 'Fixed Months Holding Strategy'

**What is the optimal number of months (1 to 12) to hold a newly IPO'd stock in order to maximize the median growth value?**

### Steps:
1. Calculate 12 future growth horizons: `future_growth_m_m = Close.shift(-21*m) / Close`.
2. Identify each stock's IPO entry day (`min_date`) and merge with the dataset.
3. Compare the median performance across holding periods 1 to 12.

In [4]:
# Compute future growth for 1 to 12 months (1 month = 21 trading days)
for m in range(1, 13):
    days = m * 21
    stocks_df[f"future_growth_{m}_m"] = stocks_df.groupby("Ticker")["Close"].shift(-days) / stocks_df["Close"]

# Identify entry points: earliest available date per ticker
min_dates = stocks_df.groupby("Ticker")["Date"].min().reset_index().rename(columns={"Date": "min_date"})
entry_df = pd.merge(stocks_df, min_dates, left_on=["Ticker", "Date"], right_on=["Ticker", "min_date"], how="inner")

growth_cols = [f"future_growth_{m}_m" for m in range(1, 13)]
summary_q3 = entry_df[growth_cols].describe().T
print("--- Summary of Holding Periods (Months 1 to 12) ---")
display(summary_q3[["count", "mean", "std", "50%"]])

medians = entry_df[growth_cols].median()
best_col = medians.idxmax()
best_month = int(re.search(r"\d+", best_col).group())
print(f"\nAnswer Q3: Optimal holding period is {best_month} month(s) with max median growth = {medians[best_col]:.4f} (Return: {(medians[best_col] - 1)*100:+.2f}%)")

--- Summary of Holding Periods (Months 1 to 12) ---


,count,mean,std,50%
future_growth_1_m,132.0,95.7515,1087.8934,0.9354
future_growth_2_m,131.0,67.9786,764.3887,0.8930
future_growth_3_m,131.0,51.2478,573.2695,0.8272
future_growth_4_m,131.0,22.6973,247.8258,0.7304
future_growth_5_m,131.0,54.2427,609.3219,0.6909
future_growth_6_m,131.0,64.0108,720.7164,0.7009
future_growth_7_m,131.0,59.2655,666.1095,0.6600
future_growth_8_m,131.0,21.4572,234.7334,0.6035
future_growth_9_m,131.0,20.0069,216.1638,0.5803
future_growth_10_m,131.0,11.2566,115.7147,0.5333



Answer Q3: Optimal holding period is 1 month(s) with max median growth = 0.9354 (Return: -6.46%)


---
## Question 4: [Strategy] Simple RSI-Based Trading Strategy

**What is the total profit (in $ thousands) you would have earned by investing $1000 every time a stock was oversold (RSI < 30)?**

### Steps:
1. Acquire precomputed dataset `data.parquet`.
2. Filter trades where `rsi < 30` between `2000-01-01` and `2025-06-01`.
3. Calculate net income: `1000 * (selected_df["growth_future_30d"] - 1).sum()`.

In [5]:
data_file = "homeworks/hw02/data.parquet" if os.path.exists("homeworks/hw02/data.parquet") else "data.parquet"
if not os.path.exists(data_file):
    file_id = "1grCTCzMZKY5sJRtdbLVCXg8JXA8VPyg-"
    print("Downloading data.parquet from Google Drive...")
    gdown.download(f"https://drive.google.com/uc?id={file_id}", data_file, quiet=False)

df_strategy = pd.read_parquet(data_file, engine="pyarrow")
df_strategy["Date_dt"] = pd.to_datetime(df_strategy["Date"])

# Filter signals
cond = (df_strategy["Date_dt"] >= "2000-01-01") & (df_strategy["Date_dt"] <= "2025-06-01") & (df_strategy["rsi"] < 30)
selected_df = df_strategy[cond].copy()

net_income = 1000.0 * (selected_df["growth_future_30d"] - 1.0).sum()
net_income_k = net_income / 1000.0
avg_return = (selected_df["growth_future_30d"] - 1.0).mean()
win_rate = (selected_df["growth_future_30d"] > 1.0).mean()

print(f"Total trades triggered: {len(selected_df)}")
print(f"Average 30-day return: {avg_return:.4%}")
print(f"Win rate: {win_rate:.2%}")
print(f"Net income: ${net_income:,.2f}")
print(f"\nAnswer Q4: Net income = ${net_income_k:,.2f} thousand (~${round(net_income_k)}k)")

Total trades triggered: 5206
Average 30-day return: 1.2640%
Win rate: 55.13%
Net income: $65,805.59

Answer Q4: Net income = $65.81 thousand (~$66k)


---
## Question 5: [Exploratory] Predicting a Positive-Return IPO

**How would you change the strategy if you want to increase profitability?**

### Key Insights & Proposed Strategy Enhancements:
1. **Avoid Day-1 Entry Hype (Wait for IPO Base):** Entering on Day 1 leads to negative median returns (-6.5% at 1 month, -51% at 12 months). Instead, let the stock consolidate for 6–8 weeks and only enter upon breakout above its consolidation high.
2. **Fundamental Quality Screening:** Filter out pre-revenue cash-burning companies. Focus on IPOs with positive operating cash flow/EBITDA and strong institutional underwriting syndicate.
3. **Insider Lockup Expiration Timing:** Exploit the 90-day / 180-day lockup cycle. Avoid holding into lockup expirations or look to enter quality stocks after the initial lockup selling wave subsides.
4. **Strict Downside Risk Control:** Set trailing stop-losses (e.g. -7% to -10%) to cut the long tail of underperforming IPOs while allowing rare multibagger outliers to run.